In [ ]:
import ipywidgets as widgets
from IPython.display import display, Javascript
import random
import asyncio
import pandas as pd
import base64
import requests
import json

# --- CONFIGURACIÓN ---
N_ENSAYOS = 36
TIEMPO_EXPOSICION = 0.2
URL_BASE_DATOS = "https://script.google.com/macros/s/AKfycbwWnN1I_fKSfuBXsN6GoS2r822FJL387AenJrQdzetMZRxJwx-dmo6J0EwoIQnFtDEo/exec"

# --- VARIABLES ---
ensayo_actual = 0
puntos_actuales = 0
resultados = []
nombre_sujeto = ""
edad_sujeto = ""

# --- INTERFAZ: FORMULARIO INICIAL ---
titulo_datos = widgets.HTML(value="<h2 style='text-align: center;'>Datos del Participante</h2>")
entrada_nombre = widgets.Text(description='Nombre:', placeholder='Ej: Juan Pérez')
entrada_edad = widgets.Text(description='Edad:', placeholder='Ej: 21')
boton_continuar = widgets.Button(description='CONTINUAR', button_style='success')
caja_datos = widgets.VBox([titulo_datos, entrada_nombre, entrada_edad, boton_continuar], layout=widgets.Layout(align_items='center', margin='30px'))

# --- INTERFAZ: EXPERIMENTO ---
instrucciones = widgets.HTML(value="<div style='text-align: center;'><h2>¡EXPERIMENTO DE PERCEPCIÓN!</h2><p>Haz clic en el cuadro de texto, ve los puntos y escribe el número + Enter.</p></div>")
pantalla = widgets.HTML(value="<div style='width: 400px; height: 300px; background-color: black; margin: auto; border-radius: 5px;'></div>")
entrada_texto = widgets.Text(placeholder='Escribe aquí...', disabled=True, layout=widgets.Layout(width='400px', margin='15px auto', display='block'))
boton_empezar = widgets.Button(description='EMPEZAR', button_style='info', layout=widgets.Layout(width='400px', margin='10px auto', display='block'))
salida = widgets.Output(layout={'width': '500px', 'margin': 'auto'})

def generar_pantalla(num_puntos):
    if num_puntos == 0: return "<div style='width: 400px; height: 300px; background-color: black; margin: auto; border-radius: 5px;'></div>"
    svg = "<svg width='400' height='300' style='background-color: black; display: block; margin: auto; border-radius: 5px;'>"
    posiciones = []
    while len(posiciones) < num_puntos:
        x, y = random.randint(30, 370), random.randint(30, 270)
        if all(((x - px)**2 + (y - py)**2)**0.5 > 30 for px, py in posiciones):
            posiciones.append((x, y))
            svg += f"<circle cx='{x}' cy='{y}' r='7' fill='white' />"
    return svg + "</svg>"

async def ciclo_ensayo():
    global puntos_actuales
    entrada_texto.disabled, entrada_texto.value = True, ''
    pantalla.value = generar_pantalla(0)
    await asyncio.sleep(random.uniform(0.6, 1.2))
    puntos_actuales = random.randint(1, 9)
    pantalla.value = generar_pantalla(puntos_actuales)
    await asyncio.sleep(TIEMPO_EXPOSICION)
    pantalla.value = generar_pantalla(0)
    entrada_texto.disabled = False
    entrada_texto.focus()

def enviar_a_nube(df):
    if URL_BASE_DATOS == "": return
    try:
        requests.post(URL_BASE_DATOS, json=df.to_dict(orient='records'))
    except:
        pass

def crear_boton_descarga():
    df = pd.DataFrame(resultados)
    enviar_a_nube(df)
    csv = df.to_csv(index=False)
    b64 = base64.b64encode(csv.encode()).decode()
    payload = f"data:text/csv;base64,{b64}"
    html = f'<a download="percepcion_{nombre_sujeto}.csv" href="{payload}" target="_blank"><button style="width:400px; height:40px; background-color:#28a745; color:white; border:none; border-radius:5px; cursor:pointer; font-weight:bold;">DESCARGAR RESULTADOS (.CSV)</button></a>'
    return widgets.HTML(html, layout=widgets.Layout(margin='20px auto', display='block'))

def registrar_respuesta(sender):
    global ensayo_actual
    if not sender.value.isdigit(): return
    resultados.append({'Nombre': nombre_sujeto, 'Edad': edad_sujeto, 'Ensayo': ensayo_actual + 1, 'Reales': puntos_actuales, 'Respuesta': int(sender.value), 'Acierto': int(sender.value) == puntos_actuales})
    ensayo_actual += 1
    if ensayo_actual < N_ENSAYOS:
        instrucciones.value = f"<h4 style='text-align: center;'>Ensayo {ensayo_actual + 1} de {N_ENSAYOS}</h4>"
        asyncio.create_task(ciclo_ensayo())
    else:
        instrucciones.value = "<h2 style='text-align: center; color: green;'>¡Prueba terminada!</h2>"
        pantalla.layout.display = 'none'
        entrada_texto.layout.display = 'none'
        with salida: display(crear_boton_descarga())

entrada_texto.on_submit(registrar_respuesta)
boton_empezar.on_click(lambda b: [setattr(boton_empezar, 'layout', widgets.Layout(display='none')), asyncio.create_task(ciclo_ensayo())])

def mostrar_experimento(b):
    global nombre_sujeto, edad_sujeto
    if entrada_nombre.value.strip() == "" or entrada_edad.value.strip() == "":
        titulo_datos.value = "<h2 style='text-align: center; color: red;'>Por favor llena tus datos</h2>"
        return
    nombre_sujeto = entrada_nombre.value.strip()
    edad_sujeto = entrada_edad.value.strip()
    caja_datos.layout.display = 'none'
    with salida: display(instrucciones, pantalla, boton_empezar, entrada_texto)

boton_continuar.on_click(mostrar_experimento)

display(salida)
with salida: display(caja_datos)